<br>

# Transform


In [ ]:
import duckdb

import tjsp_unidades.extract as ext
import tjsp_unidades.transform as tjsp

<br>

---

## Get Data


In [ ]:
quem_somos = ext.QuemSomos()
municipio = ext.Municipio()
municipio.search_batch()
municipio.detalhe_batch()

#
imovel = ext.Imovel(municipio=municipio)
imovel.search_batch()
imovel.detalhe_batch()

In [ ]:
cubo = tjsp.Transform(
    quem_somos=quem_somos,
    municipios=municipio,
    imovel=imovel,
)

In [ ]:
cubo.df_cj

<br>

---

### Quem Somos


In [ ]:
quem_somos = extract.QuemSomos()

In [ ]:
quem_somos.rajs.head(2)

In [ ]:
quem_somos.cjs.head(2)

In [ ]:
quem_somos.comarcas.head(2)

<br>

---

### Municípios


In [ ]:
municipio = extract.Municipio()

In [ ]:
# Obtem todos os IDs dos Municípios do TJSP
municipio.search_batch()

# Results
municipio.df_search.info()
municipio.df_search.head()

In [ ]:
municipio.detalhe_batch()

# Results
municipio.df_detalhes.info()
municipio.df_detalhes.head()

<br>

---

### Imóveis


In [ ]:
imovel = extract.Imovel(municipio=municipio)

In [ ]:
imovel.search(termo='forum')

In [ ]:
imovel.search_batch()
imovel.df_search.info()
imovel.df_search.head(2)

In [ ]:
imovel.detalhe_batch()
imovel.df_detalhes.info()
imovel.df_detalhes.head(2)

<br>

---

## Tratamento


In [ ]:
con = duckdb.connect()

In [ ]:
# Quem Somos
con.register("raj", quem_somos.rajs)
con.register("cj", quem_somos.cjs)
con.register("comarcas", quem_somos.comarcas)

# Município
con.register("municipio_search", municipio.df_search)
con.register("municipio_detalhes", municipio.df_detalhes)

# Imóvel
#con.register("imovel_search", imovel.df_search)
con.register("imovel_detalhes", imovel.df_detalhes)

<br>

---

### Imóveis


In [ ]:
stmt = """
    SELECT
        -- Identificador
        id_imovel,
        municipio_search.id_municipio_ibge,
        municipio_search.id_municipio_tjsp,
        imovel,

        -- Contato
        telefone,
        fax,
        email,

        --cj,        
        --entrancia, -- Acho que é atributo da Comarca
        --comarca_tjsp,

        -- Atributos
        dist_capital,
        tensao_eletrica,

        -- Endereço
        endereco_lougradouro AS lougradouro,
        endereco_cep AS cep,
        --endereco_municipio,
        municipio_search.municipio_tjsp_corrigido AS municipio_tjsp_corrigido,
        endereco_uf AS uf,

        -- Setores
        num_varas_instaladas

    FROM imovel_detalhes
    LEFT JOIN municipio_search
    ON municipio_search.municipio_tjsp = imovel_detalhes.endereco_municipio

    WHERE 1=1
        -- Confiro que o JOIN é perfeito...
        --AND municipio_search.id_municipio_ibge IS NULL
"""

# Faz a consulta
df_imoveis = con.sql(stmt).df()

# Results
df_imoveis.info()
df_imoveis.head()

<br>

---

### Municípios


In [ ]:
stmt = """
    SELECT
        DISTINCT
        -- Identificadores
        municipio_search.id_municipio_ibge,
        municipio_detalhes.id_municipio_tjsp,
        temp.id_comarca_ibge,        

        -- Outros
        municipio_detalhes.municipio_tjsp,
        municipio_search.municipio_tjsp_corrigido,
        municipio_detalhes.comarca_tjsp,
        temp.comarca_tjsp_corrigido,
        municipio_detalhes.comarca_sede        

    FROM municipio_detalhes

    LEFT JOIN municipio_search
    ON municipio_search.id_municipio_tjsp = municipio_detalhes.id_municipio_tjsp

    LEFT JOIN (
    SELECT
        DISTINCT
        -- Identificadores
        municipio_search.id_municipio_ibge AS id_comarca_ibge,
        municipio_detalhes.comarca_tjsp,

        -- Outros
        municipio_search.municipio_tjsp_corrigido AS comarca_tjsp_corrigido

    FROM municipio_detalhes

    LEFT JOIN municipio_search
    ON municipio_search.id_municipio_tjsp = municipio_detalhes.id_municipio_tjsp

    WHERE 1=1
        AND municipio_detalhes.comarca_sede = 1
        --AND municipio_detalhes.comarca_tjsp != municipio_search.municipio_tjsp_corrigido    
    ) AS temp
    ON temp.comarca_tjsp = municipio_detalhes.comarca_tjsp

    WHERE 1=1

    ORDER BY temp.id_comarca_ibge
"""

# Faz a consulta
df_municipios = con.sql(stmt).df()


# Results
df_municipios.info()
df_municipios.head()

<br>

---

### Comarcas


In [ ]:
# Tabela de Comarca
stmt = """
    SELECT
        DISTINCT
        -- Identificadores
        municipio_search.id_municipio_ibge AS id_comarca_ibge,
        comarcas.id_cj,

        -- Comarca
        municipio_detalhes.comarca_tjsp,
        municipio_search.municipio_tjsp_corrigido AS comarca_tjsp_corrigido

    FROM municipio_detalhes

    LEFT JOIN municipio_search
    ON municipio_search.id_municipio_tjsp = municipio_detalhes.id_municipio_tjsp

    LEFT JOIN comarcas
    ON comarcas.comarca_tjsp = municipio_detalhes.comarca_tjsp

    WHERE 1=1
        AND municipio_detalhes.comarca_sede = 1
        --AND municipio_detalhes.comarca_tjsp != municipio_search.municipio_tjsp_corrigido

        -- Circunscrição Judiciária é Nula
        --AND comarcas.id_cj IS NULL
"""

# Faz a consulta
df_comarcas = con.sql(stmt).df()

# Results
df_comarcas.info()
df_comarcas.head()

<br>

---

### CJs


In [ ]:
stmt = """
    SELECT
        *
    FROM cj
    WHERE 1=1
"""

# Faz a consulta
df_cjs = con.sql(stmt).df()

# Results
df_cjs.info()
df_cjs.head()

<br>

---

### RAJs


In [ ]:
stmt = """
    SELECT
        *
    FROM raj
    WHERE 1=1
"""

# Faz a consulta
df_raj = con.sql(stmt).df()

# Results
df_raj.info()
df_raj.head()